# Bitácora reto 3: Resultados

**Especialización en Inteligencia Artificial · UPTC**  
Módulo: Aprendizaje por Refuerzo y Sistemas de Recomendación

---

## QUE HAY QUE HACER
═══════════════════════════════════════════════════════════════════════════

El agente de la cuadricula tarda mucho en encontrar la meta porque durante
cientos de episodios no recibe ninguna senal util: solo el coste de cada paso.
Su trabajo es **darle una pista**, escribiendo una recompensa extra.

Rellene ``mi_moldeado``. Recibe tres cosas y devuelve un numero, que se suma a
la recompensa que el entorno ya entrega.

    anterior   la casilla donde estaba, como (fila, columna). Puede ser None
               en el primer paso.
    siguiente  la casilla a la que acaba de llegar.
    terminal   True si ``siguiente`` termina el episodio.

No hay ninguna restriccion sobre lo que puede escribir. Puede usar la
distancia a la meta, la fila, la columna, lo que se le ocurra.

═══════════════════════════════════════════════════════════════════════════
## COMO SE SABE SI FUNCIONO
═══════════════════════════════════════════════════════════════════════════

    uv run python scripts/reto3.py        entrena y mide, e imprime el veredicto
    uv run pytest tests/test_reto3.py     comprueba el contrato

**Antes de ejecutar nada, escriba su prediccion en la bitacora.** Que espera
que haga su agente. Cuantos pasos va a tardar. Que retorno va a sacar.

Aviso, y va en serio: es muy probable que su primera version obtenga un
retorno estupendo y sea un desastre. Cuando eso pase, no lo arregle todavia.
Anotelo, que de eso trata la clase.
"""

## Predicción:

En el Reto 3, al añadir una pista para el agente, predecimos que la función de recompensa modificará su comportamiento, pero podría no ayudarlo a alcanzar la meta de manera efectiva. Aunque la pista puede parecer razonable, es posible que el agente se desvíe del camino óptimo, creyendo que está mejorando su rendimiento, cuando en realidad no está logrando el objetivo deseado. Es probable que el agente se confunda y no alcance el umbral del 90% de éxito en llegar a la meta.

## Código 


from __future__ import annotations

# Estas dos las puede mover libremente.
GAMMA = 0.9      # tiene que ser el mismo descuento con el que se entrena
ESCALA = 0.5     # cuanto pesa su pista frente al coste del paso, que es -0,04

META = (0, 11)   # la esquina de arriba a la derecha de la sala de 8 x 12


def pasos_hasta_la_meta(pos: tuple[int, int]) -> int:
    """Cuantos pasos faltan hasta la meta, contando por la rejilla.

    Se la dejo hecha para que no pierda tiempo en esto. Usela o no la use.
    """
    return abs(pos[0] - META[0]) + abs(pos[1] - META[1])


def mi_moldeado(
    anterior: tuple[int, int] | None,
    siguiente: tuple[int, int],
    terminal: bool,
):
    """
    Moldeado de recompensa basado en potenciales (Potential-Based Reward Shaping)
    conforme a la teoría de Ng, Harada & Russell (1999).
    
    Conceptualmente mapea la reducción del vacío de conocimiento (distancia a la meta)
    evitando que el agente explote el sistema quedándose en bucles locales.
    """
    # 1. Obtenemos las distancias al objetivo (nuestro vacío de conocimiento)
    dist_anterior = pasos_hasta_la_meta(anterior)
    dist_siguiente = pasos_hasta_la_meta(siguiente)
    
    # 2. Definimos las funciones de potencial de estado Phi(s)
    # A menor distancia, mayor potencial (menos negativo).
    phi_anterior = -float(dist_anterior)
    phi_siguiente = -float(dist_siguiente)
    
    # 3. Factor de descuento del GridWorld
    gamma = 0.9
    
    # 4. Formulamos el shaping: F = gamma * Phi(s') - Phi(s)
    # Si es un estado terminal (llegó a la meta), el potencial futuro es 0 por definición.
    if terminal:
        shaping = 0.0 - phi_anterior
    else:
        shaping = (gamma * phi_siguiente) - phi_anterior
        
    return shaping


### Explicación

Use este código porque me conserva la política óptima: Al cumplir estrictamente con el teorema, se garantiza bajo demostración formal que la política óptima del entorno original con recompensas escasas sigue siendo la única política óptima bajo este moldeado. Además de que evita bucles infinitos: Si el agente da un paso hacia adelante y luego un paso hacia atrás, el moldeado le dará un premio positivo al acercarse, pero un castigo simétrico al alejarse, impidiendo que "estafe" al algoritmo quedándose dando vueltas en el mismo lugar para acumular recompensa infinita.

## Resultado:

  Reto 3 · Anada una pista sin estropear el problema

  Cinco semillas, 600 episodios cada una.

  Llega a la meta                           0 %      hace falta >= 90 %
  Lo que su agente CREE que saco     +529.5610      (misma politica, con su pista dentro)
  Lo que de verdad saco                -3.0000 +-0.0000   hace falta >= +0.78
  Se cree mejor en                   +532.5610      no puede pasar de +0.50
  Referencia, la politica optima       +0.8135      [+0.8110, +0.8161]

  Politica aprendida con su pista:
     < < ^ > ^ ^ ^ > ^ > > G
     < v ^ > ^ > > ^ > > > ^
     < < > > ^ > ^ ^ ^ > > >
     ^ v v < < v ^ < > > > ^
     > v v ^ > > > v ^ ^ > >
     v v v v < v > > > ^ ^ >
     v v v v < v < ^ ^ ^ ^ >
     < < < < < < ^ v > ^ ^ v

  NO SUPERADO

  1. Solo llega a la meta el 0 % de las veces. Sin ninguna pista llegaba el 100 %.
     Su pista le esta dando al agente una razon para no llegar.
  2. El retorno real (-3.0000) no alcanza el de la politica optima.
     El agente esta resolviendo un problema parecido, pero no el nuestro.
  3. Su agente se cree +532.5610 mejor de lo que es.
     Esa diferencia es lo que le esta pagando su pista por algo que no era el objetivo.

  No lo arregle todavia si es la primera vez. Anote en la bitacora que
  escribio, que esperaba y que paso. De eso trata la sesion.

Por tanto, lo anterior nos indican que el agente no logró llegar a la meta en ninguno de los 600 episodios, lo que es alarmante, ya que sin la pista, el agente alcanzaba la meta el 100% de las veces. A pesar de que el agente cree haber obtenido un retorno de +529.5610 con la nueva pista, el retorno real fue de -3.0000, lo que está muy por debajo del umbral de +0.78 requerido. Esto sugiere que la pista, en lugar de ayudar, está desviando al agente de su objetivo principal. El agente se siente más confiado, creyendo que ha mejorado su rendimiento con un retorno estimado de +532.5610, pero en realidad no puede superar el retorno de +0.50 de la política óptima. La política aprendida muestra un patrón de movimiento que no conduce a la meta, evidenciando que el agente está resolviendo un problema diferente al que se le planteó originalmente. Este resultado pone de manifiesto la importancia de diseñar adecuadamente las funciones de recompensa, ya que una pista mal concebida puede llevar a un agente a la ineficacia, generando una falsa sensación de éxito.
  

## Analísis

La mayoría implementamos una función de moldeado basada en potenciales (PBRL) que, aunque teóricamente sólida, resultó en un agente que se "cree" un éxito con un retorno marcado de +529.5610 mientras que su desempeño real es de -3.0000, evidenciando un claro caso de reward hacking donde la pista incentiva comportamientos que no conducen a la meta. Si este fuera un informe de proyecto sin supervisión, la tentación sería reportar el número positivo de +529.5610 al igual que alguien reportaría el retorno de -2.9192 en la política que "paga por estar cerca" por parecer una cifra técnica, ocultando el hecho de que el agente nunca llega a su objetivo, lo cual subraya el peligro ético y técnico de elegir métricas de conveniencia sobre la eficacia real del agente en el entorno.

## Propiedades de mi función

_________________________________________________________________________________________ test_no_paga_por_quedarse_quieto _________________________________________________________________________________________

    def test_no_paga_por_quedarse_quieto():
        """Chocar contra un muro y no moverse no puede dar dinero.
    
        Si lo da, el agente puede cobrar sin avanzar, y va a preferir eso a
        resolver el problema. Es el fallo mas comun y el mas caro.
        """
        for pos in [(4, 4), (1, 9), (6, 2)]:
>           assert extra(pos, pos) <= 1e-12, (
                f"Quedarse en {pos} paga {extra(pos, pos):+.4f}. Con eso, dar vueltas "
                "sin llegar a ninguna parte es rentable."
            )
E           Resumen: Quedarse en (4, 4) paga +1.1000. Con eso, dar vueltas sin llegar a ninguna parte es rentable.
E           assert 1.0999999999999996 <= 1e-12
E            +  where 1.0999999999999996 = extra((4, 4), (4, 4))


__________________________________________________________________________________ test_dar_una_vuelta_completa_no_deja_ganancia ___________________________________________________________________________________

    def test_dar_una_vuelta_completa_no_deja_ganancia():
        """La propiedad que lo decide todo, y se comprueba sin entrenar.
    
        Si recorrer un ciclo cerrado suma algo positivo, existe una politica que
        gana recompensa infinita sin acercarse nunca a la meta. Y el agente la va a
        encontrar, porque para eso sirve.
    
        La unica forma conocida de garantizar que esto valga cero es que la
        recompensa extra sea una **diferencia** de una funcion del estado. Ese es el
        resultado de Ng, Harada y Russell (1999), y es lo que se estudia el sabado.
        """
        total = sum(extra(a, b) for a, b in zip(CICLO, CICLO[1:]))
>       assert total <= 1e-9, (
            f"Recorrer el cuadrado {CICLO[0]} y volver deja {total:+.4f} de ganancia. "
            "Repitiendolo mil veces, su agente cobra mil veces eso sin llegar a la meta."
        )
E       Resumen: Recorrer el cuadrado (4, 4) y volver deja +4.0000 de ganancia. Repitiendolo mil veces, su agente cobra mil veces eso sin llegar a la meta.
E       assert 4.0 <= 1e-09


________________________________________________________________________________ test_con_gamma_uno_el_ciclo_suma_exactamente_cero _________________________________________________________________________________

    def test_con_gamma_uno_el_ciclo_suma_exactamente_cero():
        """Version exigente de la anterior, y la que separa lo correcto de lo que
        simplemente no es rentable todavia."""
        total = sum(extra(a, b) for a, b in zip(CICLO, CICLO[1:]))
        if abs(GAMMA - 1.0) < 1e-9:
            assert total == pytest.approx(0.0, abs=1e-9)
        else:
            # Con gamma < 1 la suma no es exactamente cero, pero tiene que ser
            # pequena comparada con lo que cuesta un paso.
>           assert abs(total) < 0.04, (
                f"El ciclo deja {total:+.4f}, comparable al coste de los pasos que lo "
                "recorren. Su pista pesa mas que el problema."
            )
E           Resumen: El ciclo deja +4.0000, comparable al coste de los pasos que lo recorren. Su pista pesa mas que el problema.
E           assert 4.0 < 0.04
E            +  where 4.0 = abs(4.0)


============================================================================================= short test summary info ==============================================================================================
FAILED tests/test_reto3.py::test_no_paga_por_quedarse_quieto - AssertionError: Quedarse en (4, 4) paga +1.1000. Con eso, dar vueltas sin llegar a ninguna parte es rentable.
FAILED tests/test_reto3.py::test_dar_una_vuelta_completa_no_deja_ganancia - AssertionError: Recorrer el cuadrado (4, 4) y volver deja +4.0000 de ganancia. Repitiendolo mil veces, su agente cobra mil veces eso sin llegar a la meta.
FAILED tests/test_reto3.py::test_con_gamma_uno_el_ciclo_suma_exactamente_cero - AssertionError: El ciclo deja +4.0000, comparable al coste de los pasos que lo recorren. Su pista pesa mas que el problema.
3 failed, 2 passed in 0.11s